In [ ]:
import pandas as pd

df = pd.read_csv("../data/clean_dataset.csv")


In [ ]:
df

In [ ]:
import numpy as np
from scipy import stats

df.drop(df.columns[0], axis=1, inplace=True)
df.dropna(inplace=True)
df = df[(np.abs(stats.zscore(df["price"])) < 3)]
#q1, q3 = df["price"].quantile([0.25, 0.75])
#iqr = q3 - q1
#df = df[df["price"].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)]

In [ ]:
df

In [ ]:
print(df.describe())
print(df.size)
print(df.ndim)
print(df.size)
print(df.info())

In [ ]:
categorial = df[["airline", "source_city", "destination_city", "class", "stops", "arrival_time", "departure_time"]]
nominal = df[["airline", "source_city", "destination_city", "arrival_time", "departure_time"]]
ordinal = df[["class", "stops"]]

quantitative = df[["duration", "days_left", "price"]]
discrete = df[["days_left"]]
continuous = df[["duration", "price"]]

cible = []

print(categorial.head())
print(ordinal.head())
print(quantitative.head())
print(categorial.describe())
print(ordinal.describe())
print(quantitative.describe())


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns


for i in nominal.columns:
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=i)
    plt.title(i)
    plt.xticks(rotation=45)
    plt.show()



In [ ]:
for i in ordinal.columns:
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=i)
    plt.title(i)
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(discrete=True,data=df, x="days_left")
plt.title("days_left")
plt.xticks(rotation=45)
plt.show()

In [ ]:
for i in continuous.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x=i)
    plt.title(i)
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(y=continuous["price"], x=ordinal["class"])
plt.title("price vs duration")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(y=continuous["price"], x=nominal["airline"], hue=ordinal["class"])
plt.title("airline vs price")
plt.xticks(rotation=45)
plt.show()


# Class and airline's influence on the price

from the two above graphs we can observe that the airline itself, has a hand on how hight the price for the ticket is, although not as much as the class, we can observe that the class has a much larger influence on the ticket price.

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(x=continuous["price"], y=ordinal["stops"], hue=ordinal["class"])
plt.title("price vs stops")
plt.xticks(rotation=45)
plt.show()


# Stops & class influence on the price
the more stops there are the higher the median ticket price is, although some outliers, most trips respect this rule.
Again like the previous observation of airline & class with prices, class here has a very high influence in the price

In [ ]:
bins = list(range(0, discrete["days_left"].max() + 5, 5))
labels = [f"{b}-{b+5}" for b in bins[:-1]]
days_binned = pd.cut(discrete["days_left"], bins=bins, labels=labels, right=False)

plt.figure(figsize=(12, 8))
sns.boxplot(x=continuous["price"], y=days_binned, hue=ordinal["class"])
plt.title("days_left vs price")
plt.xticks(rotation=45)
plt.show()

# Days_left & price
The closer the trip iss the higher the median price is

In [ ]:
import math

bins = list(range(0, math.ceil(continuous["duration"].max()) + 2, 2))
labels = [f"{b}-{b+2}" for b in bins[:-1]]
duration_binned = pd.cut(continuous["duration"], bins=bins, labels=labels, right=False)
plt.figure(figsize=(12, 8))
sns.boxplot(x=duration_binned, y=continuous["price"], hue=ordinal["class"])
plt.title("duration vs price")
plt.xticks(rotation=45)
plt.show()

# Trip duration's influence on the price
We can see a drastic spike in the price from 0-2 to 2-4 to 4-6 in the buiseness class, after that it semi stabalizes, and then rises steadelly in the 28-30 mark.

On the contrary, the econamy class has no drastic spikes, but rises steadely the higher the flight duration gets

In [ ]:

nominal_encoded = pd.get_dummies(nominal)

In [ ]:
def flatten_extend(matrix):
    flat_list = []
    for row in matrix:
        flat_list.extend(row)
    return flat_list


In [ ]:
ordinal["class"].unique()

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
columns = list(ordinal.columns.values)
ordinal = ordinal[["class", "stops"]]
encoder = OrdinalEncoder(categories=[["Economy", "Business"], ["zero", "one", "two_or_more"]])
ordinal_encoded_array = encoder.fit_transform(ordinal)

In [ ]:
ordinal_encoded_array

In [ ]:
normalized_quant = (quantitative - quantitative.min()) / (quantitative.max() - quantitative.min())

In [ ]:
ordinal_encoded = pd.DataFrame(ordinal_encoded_array)
encoded_data = pd.concat([nominal_encoded, ordinal_encoded, normalized_quant], axis=1)
encoded_data.dropna(inplace=True)
encoded_data.columns = encoded_data.columns.astype(str)

In [ ]:
encoded_data

# Machin training:

In [ ]:
from sklearn.model_selection import train_test_split

x = encoded_data.drop("price", axis=1)
y = encoded_data["price"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print(x.columns)

# Models:

In [371]:
from sklearn.dummy import DummyRegressor
dummy = DummyRegressor()
dummy.fit(x_train, y_train)

,"strategy strategy: {""mean"", ""median"", ""quantile"", ""constant""}, default=""mean""Strategy to use to generate predictions.* ""mean"": always predicts the mean of the training set* ""median"": always predicts the median of the training set* ""quantile"": always predicts a specified quantile of the training set, provided with the quantile parameter.* ""constant"": always predicts a constant value that is provided by the user.",'mean'
,"constant constant: int or float or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
,"quantile quantile: float in [0.0, 1.0], default=NoneThe quantile to predict using the ""quantile"" strategy. A quantile of0.5 corresponds to the median, while 0.0 to the minimum and 1.0 to themaximum.",None
Name,Type,Value
"constant_ constant_: ndarray of shape (1, n_outputs)Mean or median or quantile of the training targets or constant valuegiven by the user.","ndarray[float64](1, 1)",[[0.22]]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](34,)","['airline_AirAsia','airline_Air_India','airline_GO_FIRST',...,'1', 'duration','days_left']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,34
n_outputs_ n_outputs_: intNumber of outputs.,int,1


In [372]:
from sklearn.linear_model import LinearRegression

Linear = LinearRegression()
Linear.fit(x_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](34,)","[-0.01,-0.02, 0.01,..., 0.01, 0.13,-0.07]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](34,)","['airline_AirAsia','airline_Air_India','airline_GO_FIRST',...,'1', 'duration','days_left']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,0.05655
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,34
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(29)


In [373]:
from sklearn.linear_model import RidgeCV
RidgeCV = RidgeCV(alphas=[0.01, 0.05, 0.1, 0.5, 1.0], cv=5)
RidgeCV.fit(x_train, y_train)
print(RidgeCV.alpha_)

1.0


In [ ]:
from sklearn.ensemble import RandomForestRegressor
Forest = RandomForestRegressor(n_estimators=100, random_state=42)
Forest.fit(x_train, y_train)

In [ ]:
models = [dummy, Linear, RidgeCV, Forest]
for model in models:


In [369]:
def test_model(model, x_test, y_test):
    pred = model.predict(x_test)
    print(f"R²: {model.score(x_test, y_test):.4f}\n")

    comparison = pd.DataFrame({
        "real_price": y_test.values,
        "predicted_price": pred,
        "class": x_test["0"].values,
    })
    comparison["residual"] = comparison["real_price"] - comparison["predicted_price"]

    # 1) real vs predicted
    plt.figure(figsize=(8, 8))
    sns.scatterplot(data=comparison, x="real_price", y="predicted_price",
                    hue="class", alpha=0.5)
    lims = [comparison[["real_price", "predicted_price"]].min().min(),
            comparison[["real_price", "predicted_price"]].max().max()]
    plt.plot(lims, lims, color="red", linestyle="--")
    plt.xlabel("Real price")
    plt.ylabel("Predicted price")
    plt.title("Real vs predicted")
    plt.show()

    # 2) residuals
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=comparison, x="predicted_price", y="residual",
                    hue="class", alpha=0.5)
    plt.axhline(0, color="red", linestyle="--")
    plt.title("Residuals")
    plt.show()

    return comparison

[0.06082704 0.10127868 0.64780594 ... 0.56105035 0.0326638  0.0486236 ]



0.9068234658018076